# 👟 StepForward Footwear — Distributor Sales Agent
Run cells top to bottom.
- **LLM**: Zephyr-7B-Beta via HuggingFace Router (featherless-ai provider)
- **Embeddings**: all-MiniLM-L6-v2 (local ~90MB)
- **Stack**: LangGraph + LangChain 0.2+ + FastAPI + Streamlit

## STEP 1 — Install dependencies

In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-core \
    langgraph \
    'huggingface-hub>=0.25.0' \
    'sentence-transformers>=2.7.0' \
    faiss-cpu \
    'fastapi>=0.111.0' \
    'uvicorn>=0.29.0' \
    'streamlit>=1.35.0' \
    'pandas>=2.0.0' \
    'pydantic>=2.0.0' \
    python-dotenv \
    pyngrok \
    nest-asyncio \
    'httpx>=0.27.0' \
    openai
print('All packages installed.')

## STEP 2 — Upload and unzip the project

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

os.chdir('/content/footwear-agent')
print('Working directory:', os.getcwd())
!ls

## STEP 3 — Set HuggingFace token + environment paths

In [ ]:
import os

HF_TOKEN = 'hf_your_token_here'  # <-- REPLACE THIS

os.environ['HUGGINGFACEHUB_API_TOKEN'] = HF_TOKEN
os.environ['DEALERS_DB_PATH'] = '/content/footwear-agent/data/dealers.json'

with open('.env', 'w') as f:
    f.write(f'HUGGINGFACEHUB_API_TOKEN={HF_TOKEN}\n')
    f.write('DEALERS_DB_PATH=/content/footwear-agent/data/dealers.json\n')

print('Token and paths set.')

## STEP 3b — Test Zephyr endpoint
Confirms the model is reachable before running the pipeline.

In [ ]:
import os
from openai import OpenAI

token = os.environ['HUGGINGFACEHUB_API_TOKEN']

client = OpenAI(
    base_url='https://router.huggingface.co/v1',
    api_key=token,
)

print('Testing Zephyr-7B via featherless-ai...')
try:
    completion = client.chat.completions.create(
        model='HuggingFaceH4/zephyr-7b-beta:featherless-ai',
        messages=[{'role': 'user', 'content': 'Reply with only: ready'}],
        max_tokens=10,
    )
    print(f'✅ Model ready! Response: {completion.choices[0].message.content.strip()}')
except Exception as e:
    print(f'❌ Error: {e}')
    print('Check your HF token has Inference API access enabled at huggingface.co/settings/tokens')

## STEP 4 — Build RAG vectorstore
Run once — skip if faiss_index already exists.

In [ ]:
import sys
sys.path.insert(0, '/content/footwear-agent')

from backend.rag.vectorstore import build_vectorstore
build_vectorstore()
print('Vectorstore ready.')

## STEP 5 — Start FastAPI backend

In [ ]:
import nest_asyncio, threading, uvicorn, sys, time, traceback, os

nest_asyncio.apply()
sys.path.insert(0, '/content/footwear-agent')
os.environ['DEALERS_DB_PATH'] = '/content/footwear-agent/data/dealers.json'

startup_error = None

def run_api():
    global startup_error
    try:
        uvicorn.run('backend.main:app', host='0.0.0.0', port=8000, log_level='info')
    except Exception:
        startup_error = traceback.format_exc()

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
time.sleep(5)

if startup_error:
    print('FASTAPI FAILED TO START:')
    print(startup_error)
else:
    import requests
    try:
        r = requests.get('http://localhost:8000/health')
        print('API status:', r.json())
    except Exception as e:
        print('API not reachable:', e)

## STEP 6 — Launch Streamlit + public URL via ngrok

In [ ]:
from pyngrok import ngrok
import subprocess, time, os

# Kill any existing tunnels first
ngrok.kill()
time.sleep(2)

# ngrok.set_auth_token('your_ngrok_token_here')  # optional

streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'frontend/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    cwd='/content/footwear-agent',
    env={**os.environ,
         'PYTHONPATH': '/content/footwear-agent',
         'DEALERS_DB_PATH': '/content/footwear-agent/data/dealers.json'},
)

time.sleep(5)
public_url = ngrok.connect(8501)
print('=' * 60)
print(f'  STREAMLIT APP: {public_url}')
print('=' * 60)

## STEP 7 — (Optional) Quick API sanity check

In [ ]:
import requests
print('Health:', requests.get('http://localhost:8000/health').json())
dealers = requests.get('http://localhost:8000/dealers').json()
print(f'Dealers in DB: {len(dealers)}')
for d in dealers[:3]:
    print(f"  {d['dealer_id']} — {d['location']} — ${d['total_revenue']:,.0f} — {d['years_active']}yrs")

## STEP 8 — (Optional) Run pipeline directly in Python

In [ ]:
import sys, os
sys.path.insert(0, '/content/footwear-agent')
os.environ['DEALERS_DB_PATH'] = '/content/footwear-agent/data/dealers.json'

from backend.graph import run_pipeline
print('Running pipeline...')
result = run_pipeline()

leads = result['qualified_leads']
print(f'\nQualified {len(leads)} leads:')
for l in leads:
    d = l['dealer']
    print(f"  [{l['priority_tier']:4s}] {d['dealer_id']:12s} score={l['fit_score']:5.1f}  {d['location']}")

print('\nSample outreach email:')
if result['outreach_emails']:
    e = result['outreach_emails'][0]
    print(f"To: {e['dealer_id']}\nSubject: {e['subject']}")
    print(e['body'][:400], '...')

## STOP — Shutdown

In [ ]:
from pyngrok import ngrok as ng
ng.kill()
streamlit_proc.terminate()
print('Stopped.')